# Preprocesamiento
Carga del dataset, limpieza temporal y generacion de features.

In [ ]:
import pandas as pd, numpy as np
from datetime import date, timedelta

def easter_sunday(year):
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    return date(year, month, day)

def paraguay_holidays(years):
    fixed = [(1,1), (3,1), (5,1), (5,14), (5,15), (6,12), (8,15), (9,29), (12,8), (12,25)]
    out = set()
    for year in years:
        out.update(date(year, month, day) for month, day in fixed)
        easter = easter_sunday(year)
        out.add(easter - timedelta(days=3))
        out.add(easter - timedelta(days=2))
    return out

raw = pd.read_csv('../data/raw/NuevoTPIA.csv')
raw = raw.rename(columns={raw.columns[0]: 'DATETIME'})
raw['DATETIME'] = pd.to_datetime(raw['DATETIME'], utc=True, errors='coerce')
raw = raw.dropna(subset=['DATETIME', 'SIN Imputed']).sort_values('DATETIME')
raw = raw.drop_duplicates('DATETIME').set_index('DATETIME').asfreq('1h').interpolate()
idx = raw.index.tz_convert('America/Asuncion')
hour = idx.hour.to_numpy()
dow = idx.dayofweek.to_numpy()
local_dates = np.array([ts.date() for ts in idx])
holidays = paraguay_holidays(range(idx.year.min() - 1, idx.year.max() + 2))

raw['hour_sin'] = np.sin(2*np.pi*hour/24)
raw['hour_cos'] = np.cos(2*np.pi*hour/24)
raw['is_weekend'] = (dow >= 5).astype(int)
raw['is_night'] = ((hour >= 19) | (hour <= 5)).astype(int)
raw['is_morning'] = ((hour >= 6) & (hour <= 11)).astype(int)
raw['is_afternoon'] = ((hour >= 12) & (hour <= 18)).astype(int)
raw['is_peak_hour'] = (((hour >= 18) & (hour <= 22)) | ((hour >= 6) & (hour <= 8))).astype(int)
raw['is_holiday'] = np.array([d in holidays for d in local_dates], dtype=int)
raw['is_day_before_holiday'] = np.array([d + timedelta(days=1) in holidays for d in local_dates], dtype=int)
raw['is_day_after_holiday'] = np.array([d - timedelta(days=1) in holidays for d in local_dates], dtype=int)
raw['cooling_degree_24c'] = np.maximum(raw['T02M'] - 24, 0)
raw['heating_degree_18c'] = np.maximum(18 - raw['T02M'], 0)
raw['hot_afternoon'] = raw['cooling_degree_24c'] * raw['is_afternoon']
raw['hot_peak_hour'] = raw['cooling_degree_24c'] * raw['is_peak_hour']
raw['lag_1h'] = raw['SIN Imputed'].shift(1)
raw['lag_2h'] = raw['SIN Imputed'].shift(2)
raw['lag_3h'] = raw['SIN Imputed'].shift(3)
raw['lag_24h'] = raw['SIN Imputed'].shift(24)
raw['lag_48h'] = raw['SIN Imputed'].shift(48)
raw['lag_168h'] = raw['SIN Imputed'].shift(168)
raw['lag_336h'] = raw['SIN Imputed'].shift(336)
raw['rolling_6h_mean'] = raw['SIN Imputed'].shift(1).rolling(6).mean()
raw['rolling_24h_mean'] = raw['SIN Imputed'].shift(1).rolling(24).mean()
raw['wind_speed_10m'] = np.sqrt(raw['U10M']**2 + raw['V10M']**2)
processed = raw.dropna()
processed.to_csv('../data/processed/processed_dataset.csv')
processed.head()